# 🧠 Market Anomaly Detection — Training & Evaluation Pipeline

This notebook consolidates **all training and evaluation** into a single end-to-end pipeline:

| Phase | Description |
|-------|-------------|
| **Phase 1** | Train 4 Baseline anomaly models (Z-Score, Isolation Forest, LSTM Autoencoder, Prophet) |
| **Phase 2** | Train 5 Advanced anomaly models (XGBoost, HMM, TCN, VAE, Anomaly Transformer) |
| **Phase 3** | Train DL Price Forecasting models (LSTM Seq2Seq, Transformer, XGBoost Regressor) |
| **Evaluation** | Evaluate all 9 models + 2 ensembles against 24 labelled crash events |

**Assets:** SP500 · VIX · BTC · GOLD · NASDAQ · TESLA

**Outputs:**
- `backend/models/<ASSET>/scores_all.parquet` — all model scores per asset  
- `backend/models/evaluation_report.json` — full evaluation metrics  
- `backend/models/evaluation_summary.csv` — human-readable summary table

---
## ⚙️ Section 1 — Imports & Configuration

In [1]:
import json
import logging
import pickle
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT_DIR   = Path("../.."  ).resolve()   # adjust if running from a different cwd
MODELS_DIR = ROOT_DIR / "backend" / "models"
DATA_DIR   = ROOT_DIR / "backend" / "data"

sys.path.insert(0, str(ROOT_DIR / "backend" / "src"))

# ── Global settings ───────────────────────────────────────────────────────────
ASSETS     = ["SP500", "VIX", "BTC", "GOLD", "NASDAQ", "TESLA"]
WINDOW     = 7           # ±7 trading days around a crash counts as a hit
THRESHOLDS = [50.0, 60.0] # evaluate at multiple score thresholds

print(f"ROOT_DIR   : {ROOT_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

ROOT_DIR   : /Users/nilesh/Market_Anomaly_Detection_Prediction
MODELS_DIR : /Users/nilesh/Market_Anomaly_Detection_Prediction/backend/models
DATA_DIR   : /Users/nilesh/Market_Anomaly_Detection_Prediction/backend/data
PyTorch version: 2.10.0
CUDA available : False


---
## 📦 Section 2 — Local Module Imports

In [2]:
from features import load_all_features

# Baseline models
from models import (
    FEATURE_COLS,
    zscore_anomaly_score,
    train_isolation_forest, isolation_forest_score,
    train_lstm_autoencoder, lstm_anomaly_score,
    train_prophet,          prophet_anomaly_score,
    ensemble_score,         risk_label,
)

# Advanced models
from advanced_models import (
    train_xgboost, xgboost_score,
    train_hmm_regime, hmm_anomaly_score, hmm_regime_series,
    train_tcn, tcn_anomaly_score,
    dynamic_ensemble_score,
)

# Deep Learning forecasting models
from dl_models  import train_lstm_seq2seq, train_transformer
from gb_models  import train_xgboost_regressor

# Phase 2 / 3 models
from vae_model           import train_vae, vae_anomaly_score
from anomaly_transformer import train_anomaly_transformer, at_anomaly_score

print("✅ All local modules imported successfully.")

✅ All local modules imported successfully.


---
## 📂 Section 3 — Load Feature Data

In [3]:
log.info("Loading feature data...")
all_features = load_all_features()

for name, df in all_features.items():
    print(f"  {name:<8}  rows={len(df):,}  cols={len(df.columns)}")

23:16:07  INFO      Loading feature data...
23:16:07  INFO      [SP500] Loaded 4,041 rows × 27 cols from disk.
23:16:07  INFO      [VIX] Loaded 4,041 rows × 25 cols from disk.
23:16:07  INFO      [BTC] Loaded 4,164 rows × 27 cols from disk.
23:16:07  INFO      [GOLD] Loaded 4,041 rows × 27 cols from disk.
23:16:07  INFO      [NASDAQ] Loaded 4,041 rows × 27 cols from disk.
23:16:07  INFO      [TESLA] Loaded 3,919 rows × 27 cols from disk.


  SP500     rows=4,041  cols=27
  VIX       rows=4,041  cols=25
  BTC       rows=4,164  cols=27
  GOLD      rows=4,041  cols=27
  NASDAQ    rows=4,041  cols=27
  TESLA     rows=3,919  cols=27


---
## 🚀 Section 4 — Phase 1 & 2: Train Anomaly Detection Models

Trains all 9 anomaly detection models for every asset and saves `scores_all.parquet`.

In [4]:
def train_asset(name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Train all anomaly detection models for a single asset."""
    log.info(f"\n{'═'*60}")
    log.info(f"  Training all models for: {name}")
    log.info(f"{'═'*60}")
    t0 = time.time()

    # ── 1. Z-Score (no training) ──────────────────────────────────────────────
    log.info(f"[{name}] Model 1 — Z-Score (no training)")
    z_scores = zscore_anomaly_score(df)

    # ── 2. Isolation Forest ───────────────────────────────────────────────────
    log.info(f"[{name}] Model 2 — Isolation Forest")
    if_model, if_scaler = train_isolation_forest(df, name)
    if_scores = isolation_forest_score(df, if_model, if_scaler)

    # ── 3. LSTM Autoencoder ───────────────────────────────────────────────────
    log.info(f"[{name}] Model 3 — LSTM Autoencoder (PyTorch)")
    lstm_model, lstm_scaler, lstm_thresh = train_lstm_autoencoder(df, name)
    lstm_scores = lstm_anomaly_score(df, lstm_model, lstm_scaler, lstm_thresh)

    # ── 4. Prophet ────────────────────────────────────────────────────────────
    log.info(f"[{name}] Model 4 — Prophet Residual")
    prophet_model, residual_std = train_prophet(df, name)
    prophet_scores = prophet_anomaly_score(df, prophet_model, residual_std)

    # ── Baseline Ensemble (4 models) ──────────────────────────────────────────
    ens = ensemble_score(z_scores, if_scores, lstm_scores, prophet_scores)

    # ── Advanced Model A — XGBoost Classifier ────────────────────────────────
    log.info(f"[{name}] Advanced Model A — XGBoost Classifier")
    xgb_model, xgb_scaler, xgb_cols = train_xgboost(df, name)
    xgb_scores = xgboost_score(df, xgb_model, xgb_scaler, xgb_cols)

    # ── Advanced Model B — HMM Regime ────────────────────────────────────────
    log.info(f"[{name}] Advanced Model B — HMM Regime Detector")
    hmm_model, regime_map = train_hmm_regime(df, name)
    hmm_scores  = hmm_anomaly_score(df, hmm_model, regime_map)
    hmm_regimes = hmm_regime_series(df, hmm_model, regime_map)

    # ── Advanced Model C — TCN Autoencoder ───────────────────────────────────
    log.info(f"[{name}] Advanced Model C — TCN Autoencoder")
    tcn_model, tcn_scaler, tcn_cols = train_tcn(df, name)
    tcn_scores = tcn_anomaly_score(df, tcn_model, tcn_scaler, tcn_cols)

    # ── Advanced Model D — LSTM VAE ───────────────────────────────────────────
    log.info(f"[{name}] Advanced Model D — LSTM Variational Autoencoder")
    vae_model, vae_scaler, vae_thresholds = train_vae(df, name)
    vae_scores = vae_anomaly_score(df, vae_model, vae_scaler, vae_thresholds)

    # ── Advanced Model E — Anomaly Transformer ────────────────────────────────
    log.info(f"[{name}] Advanced Model E — Anomaly Transformer (ICLR 2022)")
    at_model, at_scaler, at_thresholds = train_anomaly_transformer(df, name)
    at_scores = at_anomaly_score(df, at_model, at_scaler, at_thresholds)

    # ── Dynamic Ensemble (9 models) ───────────────────────────────────────────
    adv_ens = dynamic_ensemble_score(
        z_scores, if_scores, lstm_scores, prophet_scores,
        xgb_scores, hmm_scores, tcn_scores, vae_scores, at_scores
    )

    # ── XGBoost Price Regressor ───────────────────────────────────────────────
    try:
        log.info(f"[{name}] DL Model 3 — XGBoost Price Regressor")
        xgb_price_models, xgb_price_scaler, xgb_feature_names = train_xgboost_regressor(
            df["Close"].values, df, name, lookback=30, horizon=30, test_size=0.3
        )
        meta_path = MODELS_DIR / name / "xgboost_meta.pkl"
        with open(meta_path, "wb") as f:
            pickle.dump({"models": xgb_price_models, "scaler": xgb_price_scaler,
                         "feature_names": xgb_feature_names}, f)
        log.info(f"[{name}] XGBoost Price Regressor saved to {meta_path}")
    except Exception as e:
        log.error(f"[{name}] XGBoost Price Regressor training failed: {e}")

    # ── Assemble result DataFrame ─────────────────────────────────────────────
    results = pd.DataFrame({
        "zscore_score":   z_scores,
        "iforest_score":  if_scores,
        "lstm_score":     lstm_scores,
        "prophet_score":  prophet_scores,
        "xgb_score":      (xgb_scores * 100).clip(0, 100),
        "hmm_score":      (hmm_scores * 100).clip(0, 100),
        "tcn_score":      (tcn_scores * 100).clip(0, 100),
        "vae_score":      (vae_scores * 100).clip(0, 100),
        "at_score":       (at_scores  * 100).clip(0, 100),
        "hmm_regime":     hmm_regimes,
        "ensemble_score": ens,
        "adv_ensemble":   adv_ens,
        "risk_label":     ens.apply(risk_label),
        "adv_risk":       adv_ens.apply(risk_label),
    }, index=df.index)

    # ── Save scores ───────────────────────────────────────────────────────────
    out_path = MODELS_DIR / name / "scores_all.parquet"
    results.to_parquet(out_path)
    elapsed = time.time() - t0
    log.info(f"[{name}] ✅ Done in {elapsed:.1f}s  →  {out_path}")

    top5 = results["adv_ensemble"].nlargest(5)
    log.info(f"[{name}] Top 5 anomaly dates (adv_ensemble):\n{top5.to_string()}")
    return results


# ── Run training for all assets ───────────────────────────────────────────────
anomaly_results = {}
summary = {}

for name in ASSETS:
    if name not in all_features:
        log.warning(f"No features found for {name}, skipping.")
        continue
    df = all_features[name]
    results = train_asset(name, df)
    anomaly_results[name] = results
    summary[name] = {
        "rows":          len(results),
        "baseline_anom": int((results["ensemble_score"] >= 60).sum()),
        "baseline_extr": int((results["ensemble_score"] >= 75).sum()),
        "adv_anom":      int((results["adv_ensemble"]   >= 60).sum()),
        "adv_extr":      int((results["adv_ensemble"]   >= 75).sum()),
        "max_baseline":  float(results["ensemble_score"].max()),
        "max_adv":       float(results["adv_ensemble"].max()),
        "mean_baseline": float(results["ensemble_score"].mean()),
        "mean_adv":      float(results["adv_ensemble"].mean()),
    }

# ── Training summary table ────────────────────────────────────────────────────
log.info("\n" + "═"*60)
log.info("  TRAINING COMPLETE — Summary")
log.info("═"*60)
log.info(f"  {'Asset':<8}  {'Rows':<6}  {'Baseline':<20}  {'Advanced':<20}")
log.info("  " + "─"*58)
for name, stats in summary.items():
    log.info(
        f"  {name:<8}  {stats['rows']:>5,}  "
        f"anom={stats['baseline_anom']:3d}  max={stats['max_baseline']:5.1f}  "
        f"anom={stats['adv_anom']:3d}  max={stats['max_adv']:5.1f}"
    )

pd.DataFrame(summary).T

23:16:07  INFO      
════════════════════════════════════════════════════════════
23:16:07  INFO        Training all models for: SP500
23:16:07  INFO      ════════════════════════════════════════════════════════════
23:16:07  INFO      [SP500] Model 1 — Z-Score (no training)
23:16:07  INFO      [SP500] Model 2 — Isolation Forest
23:16:07  INFO      [SP500] IF training rows: 2,486
23:16:08  INFO      [SP500] Isolation Forest saved.
23:16:08  INFO      [SP500] Model 3 — LSTM Autoencoder (PyTorch)
23:16:08  INFO      [SP500] LSTM training rows: 2,486
23:16:08  INFO      [SP500] LSTM sequences: (2457, 30, 15)
23:16:10  INFO      [SP500] LSTM epoch  10  val_loss=0.722156
23:16:13  INFO      [SP500] LSTM epoch  20  val_loss=0.585223
23:16:15  INFO      [SP500] LSTM epoch  30  val_loss=0.545474
23:16:17  INFO      [SP500] LSTM epoch  40  val_loss=0.512627
23:16:19  INFO      [SP500] LSTM epoch  50  val_loss=0.524644
23:16:19  INFO      [SP500] LSTM threshold: 0.919466
23:16:19  INFO      [SP5

[SP500] Training XGBoost regressor...
  Training quantile 0.025...
  Training quantile 0.5...
  Training quantile 0.975...


23:17:12  INFO      [SP500] XGBoost Price Regressor saved to /Users/nilesh/Market_Anomaly_Detection_Prediction/backend/models/SP500/xgboost_meta.pkl
23:17:12  INFO      [SP500] ✅ Done in 64.2s  →  /Users/nilesh/Market_Anomaly_Detection_Prediction/backend/models/SP500/scores_all.parquet
23:17:12  INFO      [SP500] Top 5 anomaly dates (adv_ensemble):
Date
2020-03-24    97.409101
2020-03-17    96.402007
2020-03-25    96.336657
2020-03-13    95.523017
2020-03-16    94.868078
23:17:12  INFO      
════════════════════════════════════════════════════════════
23:17:12  INFO        Training all models for: VIX
23:17:12  INFO      ════════════════════════════════════════════════════════════
23:17:12  INFO      [VIX] Model 1 — Z-Score (no training)
23:17:12  INFO      [VIX] Model 2 — Isolation Forest
23:17:12  INFO      [VIX] IF training rows: 2,486
23:17:12  INFO      [VIX] Isolation Forest saved.
23:17:12  INFO      [VIX] Model 3 — LSTM Autoencoder (PyTorch)
23:17:12  INFO      [VIX] LSTM train

[SP500] XGBoost training complete


23:17:14  INFO      [VIX] LSTM epoch  10  val_loss=0.567009
23:17:16  INFO      [VIX] LSTM epoch  20  val_loss=0.499159
23:17:18  INFO      [VIX] LSTM epoch  30  val_loss=0.487349
23:17:21  INFO      [VIX] LSTM epoch  40  val_loss=0.477240
23:17:23  INFO      [VIX] LSTM epoch  50  val_loss=0.464146
23:17:23  INFO      [VIX] LSTM threshold: 0.866800
23:17:23  INFO      [VIX] LSTM Autoencoder saved → lstm_autoencoder.pt
23:17:23  INFO      [VIX] Model 4 — Prophet Residual
23:17:23  INFO      [VIX] Prophet training rows: 2,486
23:17:23  INFO      Chain [1] start processing
23:17:23  INFO      Chain [1] done processing
23:17:23  INFO      [VIX] Prophet residual_std=4.0664 saved.
23:17:23  INFO      [VIX] Advanced Model A — XGBoost Classifier
23:17:23  INFO      [VIX] Crash labels: 161 positive (6.5%)
23:17:24  INFO      [VIX] XGBoost saved. Features used: 20
23:17:24  INFO      [VIX] Advanced Model B — HMM Regime Detector
23:17:24  INFO      [VIX] HMM saved. Crisis=2, Bull=0, Bear=1
23:17:

...

23:20:55  INFO      [TESLA] Isolation Forest saved.
23:20:55  INFO      [TESLA] Model 3 — LSTM Autoencoder (PyTorch)
23:20:55  INFO      [TESLA] LSTM training rows: 2,364
23:20:55  INFO      [TESLA] LSTM sequences: (2335, 30, 15)
23:20:57  INFO      [TESLA] LSTM epoch  10  val_loss=0.818826
23:20:59  INFO      [TESLA] LSTM epoch  20  val_loss=0.780011
23:21:01  INFO      [TESLA] LSTM epoch  30  val_loss=0.771490
23:21:02  INFO      [TESLA] LSTM early stop at epoch 32
23:21:02  INFO      [TESLA] LSTM threshold: 1.084246
23:21:02  INFO      [TESLA] LSTM Autoencoder saved → lstm_autoencoder.pt
23:21:02  INFO      [TESLA] Model 4 — Prophet Residual
23:21:02  INFO      [TESLA] Prophet training rows: 2,364
23:21:02  INFO      Chain [1] start processing
23:21:02  INFO      Chain [1] done processing
23:21:02  INFO      [TESLA] Prophet residual_std=1.7525 saved.
23:21:02  INFO      [TESLA] Advanced Model A — XGBoost Classifier
23:21:02  INFO      [TESLA] Crash labels: 28 positive (1.2%)
23:21:0

[TESLA] Training XGBoost regressor...
  Training quantile 0.025...
  Training quantile 0.5...


23:21:47  INFO      [TESLA] XGBoost Price Regressor saved to /Users/nilesh/Market_Anomaly_Detection_Prediction/backend/models/TESLA/xgboost_meta.pkl
23:21:47  INFO      [TESLA] ✅ Done in 52.0s  →  /Users/nilesh/Market_Anomaly_Detection_Prediction/backend/models/TESLA/scores_all.parquet
23:21:47  INFO      [TESLA] Top 5 anomaly dates (adv_ensemble):
Date
2021-12-03    95.432456
2022-05-05    95.365300
2022-05-09    95.075390
2022-05-13    94.744773
2022-05-16    94.234716
23:21:47  INFO      
════════════════════════════════════════════════════════════
23:21:47  INFO        TRAINING COMPLETE — Summary
23:21:47  INFO      ════════════════════════════════════════════════════════════
23:21:47  INFO        Asset     Rows    Baseline              Advanced            
23:21:47  INFO        ──────────────────────────────────────────────────────────
23:21:47  INFO        SP500     4,041  anom=218  max= 93.8  anom=183  max= 97.4
23:21:47  INFO        VIX       4,041  anom= 78  max= 88.7  anom= 9

  Training quantile 0.975...
[TESLA] XGBoost training complete


,rows,baseline_anom,baseline_extr,adv_anom,adv_extr,max_baseline,max_adv,mean_baseline,mean_adv
SP500,4041.0,218.0,72.0,183.0,66.0,93.755203,97.409101,27.735356,21.077056
VIX,4041.0,78.0,30.0,99.0,41.0,88.651500,96.922942,21.951983,19.048614
BTC,4164.0,516.0,27.0,32.0,16.0,90.810179,96.162915,32.077333,24.884097
GOLD,4041.0,72.0,29.0,49.0,0.0,93.817303,74.051884,24.260883,17.025650
NASDAQ,4041.0,582.0,275.0,292.0,136.0,93.948247,96.872212,32.581540,24.668694
TESLA,3919.0,1457.0,327.0,244.0,121.0,94.125339,95.432456,39.541711,31.420527


---
## 🔮 Section 5 — Phase 3: Train DL Price Forecasting Models

Trains **LSTM Seq2Seq** and **Transformer** models using the full 28-feature multivariate dataframe.  
Requires `scores_all.parquet` to be present (generated in Section 4).

In [5]:
def train_dl_models_for_asset(asset: str, df: pd.DataFrame):
    """Train multivariate LSTM Seq2Seq and Transformer for a single asset."""
    df = df.copy()
    asset_dir = MODELS_DIR / asset
    asset_dir.mkdir(parents=True, exist_ok=True)

    # ── Inject anomaly risk as 28th feature ───────────────────────────────────
    scores_path = asset_dir / "scores_all.parquet"
    if scores_path.exists():
        scores_df = pd.read_parquet(scores_path)
        df["anomaly_risk"] = scores_df["adv_ensemble"] / 100.0
        log.info(f"[{asset}] Anomaly risk injected. Total features: {len(df.columns)}")
    else:
        log.warning(f"[{asset}] scores_all.parquet not found — using 0.0 for anomaly_risk")
        df["anomaly_risk"] = 0.0

    log.info(f"\n{'═'*60}")
    log.info(f"  Training DL Forecasting Models for: {asset}")
    log.info(f"{'═'*60}")

    # ── DL Model 1: LSTM Seq2Seq ──────────────────────────────────────────────
    try:
        log.info(f"[{asset}] Training LSTM Seq2Seq (Multivariate)")
        lstm_model, lstm_scaler, lstm_scaler_x, feature_cols = train_lstm_seq2seq(
            df, asset, lookback=30, horizon=30, epochs=20, batch_size=32
        )
        torch.save(lstm_model, asset_dir / "lstm_seq2seq.pt")
        with open(asset_dir / "lstm_seq2seq_meta.pkl", "wb") as f:
            pickle.dump({"scaler": lstm_scaler, "scaler_x": lstm_scaler_x,
                         "feature_cols": feature_cols, "mode": "returns"}, f)
        log.info(f"[{asset}] ✅ LSTM Seq2Seq saved.")
    except Exception as e:
        log.error(f"[{asset}] LSTM training failed: {e}")

    # ── DL Model 2: Transformer ───────────────────────────────────────────────
    try:
        log.info(f"[{asset}] Training Transformer (Multivariate)")
        trans_model, trans_scaler, trans_scaler_x, t_feature_cols = train_transformer(
            df, asset, lookback=30, horizon=30, epochs=20, batch_size=32
        )
        torch.save(trans_model, asset_dir / "transformer.pt")
        with open(asset_dir / "transformer_meta.pkl", "wb") as f:
            pickle.dump({"scaler": trans_scaler, "scaler_x": trans_scaler_x,
                         "feature_cols": t_feature_cols, "mode": "returns"}, f)
        log.info(f"[{asset}] ✅ Transformer saved.")
    except Exception as e:
        log.error(f"[{asset}] Transformer training failed: {e}")


# ── Run DL training for all assets ───────────────────────────────────────────
for asset in ASSETS:
    if asset not in all_features:
        log.warning(f"No features for {asset}, skipping DL training.")
        continue
    train_dl_models_for_asset(asset, all_features[asset])

print("\n✅ Phase 3 DL model training complete.")

23:21:47  INFO      [SP500] Anomaly risk injected. Total features: 28
23:21:47  INFO      
════════════════════════════════════════════════════════════
23:21:47  INFO        Training DL Forecasting Models for: SP500
23:21:47  INFO      ════════════════════════════════════════════════════════════
23:21:47  INFO      [SP500] Training LSTM Seq2Seq (Multivariate)


[SP500] LSTM Epoch 10/20 - Train: 0.130459, Val: 0.189820


23:22:22  INFO      [SP500] ✅ LSTM Seq2Seq saved.
23:22:22  INFO      [SP500] Training Transformer (Multivariate)


[SP500] LSTM early stopping at epoch 11
[SP500] Transformer Epoch 10/20 - Train: 0.142817, Val: 0.167458


23:23:43  INFO      [SP500] ✅ Transformer saved.
23:23:43  INFO      [VIX] Anomaly risk injected. Total features: 26
23:23:43  INFO      
════════════════════════════════════════════════════════════
23:23:43  INFO        Training DL Forecasting Models for: VIX
23:23:43  INFO      ════════════════════════════════════════════════════════════
23:23:44  INFO      [VIX] Training LSTM Seq2Seq (Multivariate)


...

[NASDAQ] Transformer early stopping at epoch 13
[TESLA] LSTM Epoch 10/20 - Train: 0.141068, Val: 0.193838


23:33:02  INFO      [TESLA] ✅ LSTM Seq2Seq saved.
23:33:02  INFO      [TESLA] Training Transformer (Multivariate)


[TESLA] LSTM Epoch 20/20 - Train: 0.109967, Val: 0.202793
[TESLA] Transformer Epoch 10/20 - Train: 0.151706, Val: 0.188180


23:34:09  INFO      [TESLA] ✅ Transformer saved.


[TESLA] Transformer early stopping at epoch 12

✅ Phase 3 DL model training complete.


---
## 📊 Section 6 — Evaluation: All Models vs Crash Labels

Evaluates all 9 anomaly models + 2 ensembles against the **24 labelled crash events** in `crash_labels.json`.  
Outputs:
- `evaluation_report.json` — full per-asset per-model metrics  
- `evaluation_summary.csv` — flat table for quick inspection

In [6]:
# ── Helper: load crash labels ─────────────────────────────────────────────────
def load_crash_labels() -> list[dict]:
    path = DATA_DIR / "crash_labels.json"
    with open(path) as f:
        data = json.load(f)
    # Support both flat list and nested {"events": [...]} format
    return data if isinstance(data, list) else data.get("events", [])


# ── Helper: ground-truth binary vector ───────────────────────────────────────
def build_ground_truth(index: pd.Index, crash_dates: list[str],
                        window: int = WINDOW) -> np.ndarray:
    """Binary array: 1 if date is within `window` days of a crash."""
    gt = np.zeros(len(index), dtype=int)
    idx_dates = pd.to_datetime(index)
    for cd in crash_dates:
        crash_dt = pd.Timestamp(cd)
        for i, d in enumerate(idx_dates):
            if abs((d - crash_dt).days) <= window:
                gt[i] = 1
    return gt


# ── Helper: compute all metrics for one model ─────────────────────────────────
def evaluate_scores(scores: pd.Series, gt: np.ndarray,
                    crash_dates: list[str], threshold: float = 50.0) -> dict:
    """Returns full metric dict for one score column vs ground truth."""
    s    = scores.fillna(0).values / 100.0
    pred = (scores.fillna(0).values >= threshold).astype(int)

    result = {"threshold": threshold}
    result["precision"] = round(float(precision_score(gt, pred, zero_division=0)), 4)
    result["recall"]    = round(float(recall_score(gt, pred, zero_division=0)), 4)
    result["f1"]        = round(float(f1_score(gt, pred, zero_division=0)), 4)

    if gt.sum() > 0 and gt.sum() < len(gt):
        result["roc_auc"] = round(float(roc_auc_score(gt, s)), 4)
    else:
        result["roc_auc"] = None

    hits, total = 0, 0
    lead_days   = []
    idx_dates   = pd.to_datetime(scores.index)
    for cd in crash_dates:
        crash_dt = pd.Timestamp(cd)
        if crash_dt < idx_dates[0] or crash_dt > idx_dates[-1]:
            continue
        total += 1
        mask = (
            (idx_dates >= crash_dt - pd.Timedelta(days=WINDOW)) &
            (idx_dates <= crash_dt + pd.Timedelta(days=WINDOW))
        )
        if (scores[mask] >= threshold).any():
            hits += 1
            before  = scores[idx_dates <= crash_dt]
            flagged = before[before >= threshold]
            if not flagged.empty:
                lead = (crash_dt - pd.Timestamp(flagged.index[-1])).days
                if 0 <= lead <= 30:
                    lead_days.append(lead)

    result["hit_rate"]         = round(hits / total, 4) if total else 0.0
    result["crashes_detected"] = hits
    result["crashes_in_range"] = total
    result["avg_lead_days"]    = round(float(np.mean(lead_days)), 1) if lead_days else 0.0
    return result


print("✅ Evaluation helpers defined.")

✅ Evaluation helpers defined.


In [7]:
# ── Run evaluation ────────────────────────────────────────────────────────────
crashes     = load_crash_labels()
crash_dates = [c["date"] for c in crashes]
log.info(f"Loaded {len(crash_dates)} crash events")

SCORE_COLS = [
    "zscore_score", "iforest_score", "lstm_score", "prophet_score",
    "xgb_score", "hmm_score", "tcn_score", "vae_score", "at_score",
    "ensemble_score", "adv_ensemble",
]
COL_LABELS = {
    "zscore_score":   "Z-Score",
    "iforest_score":  "Isolation Forest",
    "lstm_score":     "LSTM Autoencoder",
    "prophet_score":  "Prophet Residual",
    "xgb_score":      "XGBoost Classifier",
    "hmm_score":      "HMM Regime",
    "tcn_score":      "TCN Autoencoder",
    "vae_score":      "VAE Autoencoder",
    "at_score":       "Anomaly Transformer",
    "ensemble_score": "Baseline Ensemble",
    "adv_ensemble":   "Advanced Ensemble",
}

report = {}
rows   = []

for asset in ASSETS:
    scores_path = MODELS_DIR / asset / "scores_all.parquet"
    if not scores_path.exists():
        log.warning(f"[{asset}] scores_all.parquet not found — skipping")
        continue

    df = pd.read_parquet(scores_path)
    df.index = pd.to_datetime(df.index)
    gt = build_ground_truth(df.index, crash_dates)
    log.info(f"[{asset}] rows={len(df):,}  crash_windows={gt.sum():,}")

    report[asset] = {}
    for col in SCORE_COLS:
        if col not in df.columns:
            log.warning(f"  [{asset}] {col} not in scores — skipping")
            continue

        # Evaluate at each threshold, keep best F1
        best_metrics, best_f1 = None, -1
        for thresh in THRESHOLDS:
            metrics = evaluate_scores(df[col], gt, crash_dates, threshold=thresh)
            if metrics["f1"] > best_f1:
                best_f1, best_metrics = metrics["f1"], metrics

        report[asset][col] = best_metrics
        label = COL_LABELS[col]
        log.info(
            f"  [{asset}] {label:<22}  "
            f"P={best_metrics['precision']:.3f}  R={best_metrics['recall']:.3f}  "
            f"F1={best_metrics['f1']:.3f}  AUC={best_metrics.get('roc_auc') or 0:.3f}  "
            f"Hit={best_metrics['hit_rate']:.2f}  Lead={best_metrics['avg_lead_days']:.1f}d"
        )
        rows.append({
            "asset":            asset,
            "model":            label,
            "threshold":        best_metrics["threshold"],
            "f1":               best_metrics["f1"],
            "precision":        best_metrics["precision"],
            "recall":           best_metrics["recall"],
            "roc_auc":          best_metrics.get("roc_auc") or 0,
            "hit_rate":         best_metrics["hit_rate"],
            "crashes_detected": best_metrics["crashes_detected"],
            "crashes_in_range": best_metrics["crashes_in_range"],
            "avg_lead_days":    best_metrics["avg_lead_days"],
        })

# ── Save JSON report ──────────────────────────────────────────────────────────
report_path = MODELS_DIR / "evaluation_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
log.info(f"\nSaved → {report_path}")

# ── Save CSV summary ──────────────────────────────────────────────────────────
df_sum = pd.DataFrame(rows)
summary_path = MODELS_DIR / "evaluation_summary.csv"
df_sum.to_csv(summary_path, index=False)
log.info(f"Saved → {summary_path}")

print(f"\n✅ Evaluation complete. {len(rows)} model×asset pairs evaluated.")

23:34:09  INFO      Loaded 24 crash events
23:34:09  INFO      [SP500] rows=4,041  crash_windows=259
23:34:09  INFO        [SP500] Z-Score                 P=0.000  R=0.000  F1=0.000  AUC=0.617  Hit=0.00  Lead=0.0d
23:34:09  INFO        [SP500] Isolation Forest        P=0.000  R=0.000  F1=0.000  AUC=0.784  Hit=0.00  Lead=0.0d
23:34:09  INFO        [SP500] LSTM Autoencoder        P=0.000  R=0.000  F1=0.000  AUC=0.730  Hit=0.00  Lead=0.0d
23:34:09  INFO        [SP500] Prophet Residual        P=0.000  R=0.000  F1=0.000  AUC=0.631  Hit=0.00  Lead=0.0d
23:34:09  INFO        [SP500] XGBoost Classifier      P=0.592  R=0.695  F1=0.639  AUC=0.917  Hit=0.83  Lead=0.0d
23:34:09  INFO        [SP500] HMM Regime              P=0.207  R=0.517  F1=0.295  AUC=0.803  Hit=0.71  Lead=0.0d
23:34:09  INFO        [SP500] TCN Autoencoder         P=0.118  R=0.035  F1=0.054  AUC=0.680  Hit=0.08  Lead=0.0d
23:34:09  INFO        [SP500] VAE Autoencoder         P=0.121  R=0.266  F1=0.167  AUC=0.677  Hit=0.33  Lead=


✅ Evaluation complete. 66 model×asset pairs evaluated.


---
## 📋 Section 7 — Results Summary

In [8]:
# ── Ensemble performance by asset ─────────────────────────────────────────────
print("=" * 80)
print("  RESULTS BY ENSEMBLE TYPE")
print("=" * 80)

for ens_name in ["Baseline Ensemble", "Advanced Ensemble"]:
    ens_df = df_sum[df_sum["model"] == ens_name]
    if ens_df.empty:
        continue
    print(f"\n  ── {ens_name} ──")
    for _, row in ens_df.iterrows():
        print(
            f"    {row['asset']:<8}  "
            f"P={row['precision']:.3f}  R={row['recall']:.3f}  "
            f"F1={row['f1']:.3f}  AUC={row['roc_auc']:.3f}  "
            f"Hit={int(row['crashes_detected'])}/{int(row['crashes_in_range'])}  "
            f"Lead={row['avg_lead_days']:.1f}d"
        )
    print(
        f"    {'AVERAGE':<8}  "
        f"P={ens_df['precision'].mean():.3f}  R={ens_df['recall'].mean():.3f}  "
        f"F1={ens_df['f1'].mean():.3f}  AUC={ens_df['roc_auc'].mean():.3f}  "
        f"Hit={ens_df['hit_rate'].mean():.1%}"
    )

  RESULTS BY ENSEMBLE TYPE

  ── Baseline Ensemble ──
    SP500     P=0.221  R=0.313  F1=0.259  AUC=0.764  Hit=14/24  Lead=0.2d
    VIX       P=0.351  R=0.178  F1=0.236  AUC=0.767  Hit=12/24  Lead=0.0d
    BTC       P=0.086  R=0.261  F1=0.129  AUC=0.575  Hit=10/19  Lead=0.4d
    GOLD      P=0.057  R=0.035  F1=0.043  AUC=0.608  Hit=5/24  Lead=1.5d
    NASDAQ    P=0.167  R=0.374  F1=0.231  AUC=0.727  Hit=12/24  Lead=0.0d
    TESLA     P=0.096  R=0.565  F1=0.164  AUC=0.645  Hit=13/23  Lead=0.0d
    AVERAGE   P=0.163  R=0.288  F1=0.177  AUC=0.681  Hit=48.1%

  ── Advanced Ensemble ──
    SP500     P=0.346  R=0.510  F1=0.412  AUC=0.922  Hit=19/24  Lead=0.0d
    VIX       P=0.502  R=0.471  F1=0.486  AUC=0.901  Hit=17/24  Lead=0.2d
    BTC       P=0.162  R=0.151  F1=0.157  AUC=0.619  Hit=5/19  Lead=0.6d
    GOLD      P=0.221  R=0.097  F1=0.134  AUC=0.662  Hit=4/24  Lead=0.0d
    NASDAQ    P=0.256  R=0.560  F1=0.351  AUC=0.860  Hit=17/24  Lead=0.0d
    TESLA     P=0.266  R=0.262  F1=0.264  AUC

In [9]:
# ── Best individual model per asset ───────────────────────────────────────────
print("=" * 80)
print("  BEST INDIVIDUAL MODEL PER ASSET (by F1)")
print("=" * 80)
for asset in ASSETS:
    sub = df_sum[df_sum["asset"] == asset]
    if sub.empty:
        continue
    best = sub.loc[sub["f1"].idxmax()]
    print(
        f"  {asset:<8}  {best['model']:<22}  "
        f"F1={best['f1']:.3f}  AUC={best['roc_auc']:.3f}  "
        f"Hit={int(best['crashes_detected'])}/{int(best['crashes_in_range'])}"
    )

  BEST INDIVIDUAL MODEL PER ASSET (by F1)
  SP500     XGBoost Classifier      F1=0.639  AUC=0.917  Hit=20/24
  VIX       XGBoost Classifier      F1=0.640  AUC=0.926  Hit=20/24
  BTC       HMM Regime              F1=0.233  AUC=0.667  Hit=11/19
  GOLD      XGBoost Classifier      F1=0.271  AUC=0.659  Hit=4/24
  NASDAQ    XGBoost Classifier      F1=0.586  AUC=0.868  Hit=18/24
  TESLA     XGBoost Classifier      F1=0.307  AUC=0.766  Hit=9/23


In [10]:
# ── Full pivoted table: F1 by model × asset ───────────────────────────────────
pivot = df_sum.pivot_table(index="model", columns="asset", values="f1", aggfunc="mean")
pivot["MEAN"] = pivot.mean(axis=1)
pivot = pivot.sort_values("MEAN", ascending=False)

display(pivot.style
    .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
    .format("{:.3f}")
    .set_caption("F1 Score by Model × Asset (green = better)"))

asset,BTC,GOLD,NASDAQ,SP500,TESLA,VIX,MEAN
model,,,,,,,
XGBoost Classifier,0.092,0.271,0.586,0.639,0.307,0.640,0.423
Advanced Ensemble,0.157,0.134,0.351,0.412,0.264,0.486,0.301
HMM Regime,0.233,0.144,0.267,0.295,0.166,0.232,0.223
Baseline Ensemble,0.129,0.043,0.231,0.259,0.164,0.236,0.177
VAE Autoencoder,0.138,0.062,0.186,0.167,0.161,0.202,0.153
Anomaly Transformer,0.139,0.026,0.158,0.220,0.157,0.191,0.148
TCN Autoencoder,0.082,0.083,0.193,0.054,0.191,0.083,0.114
Isolation Forest,0.000,0.000,0.000,0.000,0.000,0.000,0.000
LSTM Autoencoder,0.000,0.000,0.000,0.000,0.000,0.000,0.000
